# Foco-LLM: seletor aritmético 3B no Kaggle

Ative **GPU T4 x2** e **Internet** nas configurações do notebook. O experimento usa apenas a primeira T4, com QLoRA 4-bit. A avaliação usa os mesmos 40 problemas diagnósticos congelados do experimento local; nenhum gabarito de validação entra no treino.

Execute as células na ordem. O treino padrão tem 320 atualizações e checkpoint a cada 40. Os resultados ficam em `/kaggle/working/foco-output`; salve uma versão do notebook ou baixe essa pasta antes de encerrar a sessão. A duração real e a acurácia dependem da execução e não são estimadas pelo notebook. O [modelo 3B](https://huggingface.co/Qwen/Qwen2.5-3B-Instruct) tem licença Qwen Research para uso de pesquisa; não redistribua os pesos. A comparação com o treino local 1,5B também muda a precisão (BF16 versus NF4/FP16), então não isola apenas o efeito do número de parâmetros.

In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path

if sys.version_info[:2] != (3, 12):
    raise RuntimeError(f"This locked project requires Python 3.12; found {sys.version}")
if shutil.which("poetry") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "poetry==2.4.1"], check=True)
repository = Path("/kaggle/working/foco-llm")
if not repository.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "--branch",
            "feat/task-balanced-training",
            "--single-branch",
            "https://github.com/Mathwesm/foco-llm.git",
            str(repository),
        ],
        check=True,
    )
subprocess.run(["git", "rev-parse", "HEAD"], cwd=repository, check=True)
subprocess.run(
    ["poetry", "install", "--only", "main,inference", "--no-interaction"],
    cwd=repository,
    check=True,
)

## 1. Preparar os dados e medir o modelo 3B sem treino

A geração da base só continua se o hash coincidir com o experimento original. O baseline controla a capacidade do modelo maior antes do ajuste.

In [ ]:
output = Path("/kaggle/working/foco-output")
command = ["poetry", "run", "python", "scripts/run_kaggle_selection.py"]
for mode in ("prepare", "baseline"):
    subprocess.run([*command, mode, "--output", str(output)], cwd=repository, check=True)

## 2. Treinar o seletor com QLoRA

A saída supervisionada contém só IDs dos fatos relevantes; o modelo não recebe a resposta aritmética. Se a sessão cair, rode esta célula outra vez no mesmo diretório restaurado: o treinamento retoma do último checkpoint íntegro. Para uma prova curta, substitua `320` por `2`, mas use uma pasta de saída diferente e não interprete esse smoke test como resultado científico.

In [ ]:
subprocess.run(
    [*command, "train", "--output", str(output), "--steps", "320", "--checkpoint-every", "40"],
    cwd=repository,
    check=True,
)
training_runs = sorted((output / "training").glob("*/summary.json"))
if len(training_runs) != 1:
    raise RuntimeError(f"Expected one completed training run, found {len(training_runs)}")
adapter = training_runs[0].parent / "step-0320"
if not (adapter / "complete.json").is_file():
    raise RuntimeError("The final adapter checkpoint is missing")
print(f"Final adapter: {adapter}")

## 3. Avaliar o adaptador e comparar

O arquivo `scores.json` preserva cada resposta bruta. Compare principalmente `similar-4`, dez casos com quatro distratores parecidos; as demais condições servem como diagnóstico. Trata-se de uma extensão exploratória do TCC, pois os templates sintéticos ainda permitem atalhos lexicais.

In [ ]:
import json

subprocess.run(
    [*command, "adapted", "--output", str(output), "--adapter", str(adapter)],
    cwd=repository,
    check=True,
)
summaries = {}
for name in ("baseline", "adapted"):
    files = sorted((output / name).glob("*/summary.json"))
    if len(files) != 1:
        raise RuntimeError(f"Expected one {name} summary, found {len(files)}")
    summaries[name] = json.loads(files[0].read_text(encoding="utf-8"))
    print(name, summaries[name]["groups"]["similar-4"])

In [ ]:
import matplotlib.pyplot as plt

forms = ("text-1", "text-2", "text-4", "similar-4")
fig, ax = plt.subplots(figsize=(8, 4.5))
positions = list(range(len(forms)))
for offset, name, color in ((-0.18, "baseline", "#0072B2"), (0.18, "adapted", "#D55E00")):
    values = [summaries[name]["groups"][form]["selection_exact"] for form in forms]
    ax.bar(
        [position + offset for position in positions], values, width=0.36, label=name, color=color
    )
ax.set_xticks(positions, forms)
ax.set_ylim(0, 10)
ax.set_ylabel("Seleções exatas (de 10 casos)")
ax.set_xlabel("Condição diagnóstica")
ax.set_title("Qwen 2.5 3B: seleção de evidências antes e depois de QLoRA")
ax.legend()
fig.tight_layout()
fig.savefig(output / "selection-comparison.png", dpi=160, bbox_inches="tight")
plt.show()